In [1]:
import os
from pathlib import Path
import json
from jsonargparse import CLI
import boto3

import time
from copy import deepcopy
import threading
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline
from peft import PeftModel
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

def get_batch_response(text, base_model, model_path, temperature, max_tokens, EXSTING):
    """
    Modified get_response function to use local model instead of API calls.
    model_path: path to the local model directory
    tokenizer and model: optional pre-loaded tokenizer and model objects
    """
    
    while True:
        try:
            # content = prompt.format(text)
            
            # Load model and tokenizer if not provided (for thread safety)
            
            # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

            model = LLM(model=base_model, 
                        enable_lora=True,
                        tensor_parallel_size=8,
                        dtype="bfloat16")

            lora = LoRARequest(lora_name="llama-3.2-instruct",
                                lora_int_id=1,
                                lora_path=model_path)

            sampling_params = SamplingParams(max_tokens=max_tokens,
                                            temperature=temperature,
                                            top_p=0.9)

            outputs = model.generate(text, sampling_params, lora_request=lora)

            results = [output.outputs[0].text for output in outputs]
            
            ##############################################
            ################# Approach 1 #################
            ##############################################

            # # Load tokenizer
            # tokenizer = AutoTokenizer.from_pretrained(base_model)
            # base_model = AutoModelForCausalLM.from_pretrained(base_model,  
            #                     torch_dtype=torch.float16,
            #                     device_map='auto'
            #                     )

            # # Load LoRA adapter on top of base model
            # model = PeftModel.from_pretrained(base_model, model_path)

            # if tokenizer.pad_token is None:
            #     tokenizer.pad_token = tokenizer.eos_token
            
            # tokenizer.padding_side = "left"
            
            # model = model.merge_and_unload()

            # model.eval()
            # results=[]

            # pipe = TextGenerationPipeline(model=model, tokenizer=tokenizer)

            # results = pipe(text, max_new_tokens=max_tokens,
            #                 temperature=temperature,
            #                 do_sample=temperature > 0,
            #                 pad_token_id=tokenizer.pad_token_id,
            #                 eos_token_id=tokenizer.eos_token_id,
            #                 use_cache=True,
            #                 batch_size=len(text)
            #             )

            # results = [result[0]['generated_text'][len(prompt):].strip() if result[0]['generated_text'].startswith(prompt) else result[0]['generated_text']
            #                     for prompt, result in zip(text, results)]
            

            ##############################################
            ################# Approach 2 #################
            ##############################################

            # for content in text:
            #     # Tokenize input
            #     inputs = tokenizer(
            #         content,
            #         return_tensors="pt",
            #         truncation=True,
            #         max_length=512,
            #         padding=True
            #     )
            
            #     # Move to device
            #     device = next(model.parameters()).device
            #     inputs = {k: v.to(device) for k, v in inputs.items()}
            
            #     # Generate response
            #     with torch.no_grad():
            #         outputs = model.generate(
            #             **inputs,
            #             max_new_tokens=max_tokens,
            #             temperature=temperature,
            #             do_sample=temperature > 0,
            #             pad_token_id=tokenizer.pad_token_id,
            #             eos_token_id=tokenizer.eos_token_id,
            #             use_cache=True
            #         )
            
            #     # Decode response (only the new tokens)
            #     input_length = inputs['input_ids'].shape[1]
            #     generated_tokens = outputs[0][input_length:]
            #     response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            #     results.append(response_text)
            
            
            return results
            
            
        except Exception as e:
            if e == KeyboardInterrupt:
                raise e
            print(f"Error: {e}")
            time.sleep(2)
            continue

        break


def main(from_json: str = None, to_json: str = None, prompt: str = None, base_model: str = 'llama-3.1-instruct',
         model_path: str = 'llama-3.1-instruct', temperature: float = 0, max_tokens: int = 512, 
         batch_size: int = 8, n_print: int = 100, n_samples: int = -1, 
         input_field: str = 'input', existing_json: str = None):
    EXSTING = {}
    if existing_json is not None:
        with open(existing_json, 'r') as f:
            for l in f.readlines():
                d = json.loads(l)
                if d['resp'] != 'API Failed':
                    EXSTING[d['prompt']] = d
    
    
    path = Path(to_json)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.touch()

    with open(from_json, "r") as fr, open(to_json, 'w') as fw:

        results = []

        lines = fr.readlines()
        total_lines = min(len(lines), n_samples) if n_samples > 0 else len(lines)
        start_time = time.time()
        
        for i in range(0, len(lines), batch_size):
           
            batch = [prompt.format(json.loads(lines[i+j])[input_field]) for j in range(min(batch_size, len(lines)-i))]
            
            batch_results = get_batch_response(
                                batch, base_model, model_path, temperature, max_tokens, EXSTING
                            )
            
            for result in batch_results:
                fw.write(json.dumps(result) + '\n')

            if i % n_print == 0:
                print(f'Time elapsed: {time.time() - start_time:.2f} sec. {i+8} / {total_lines} samples generated. ')

main(from_json='testsets/inspired/test_clean.jsonl',
    to_json='test_res/inspired/llama-3.2-instruct/inspired_test_clean.jsonl',
    prompt="Pretend you are a movie recommender system. I will give you a conversation between a user and you (a recommender system). Based on the conversation, you reply with a list of 20 recommendations in the format of '1. [Movie Name]\n 2. [Movie Name]\n ...' with no extra sentences no user reponse. Here is the conversation: {}",
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    model_path='../outputs/sft/inspired/test',
    temperature=0.1,
    max_tokens=512,
    n_print=1,
    n_samples=-1)

INFO 06-27 05:22:16 [__init__.py:244] Automatically detected platform cuda.
INFO 06-27 05:22:25 [config.py:823] This model supports multiple tasks: {'score', 'embed', 'generate', 'reward', 'classify'}. Defaulting to 'generate'.
INFO 06-27 05:22:25 [config.py:1946] Defaulting to use mp for distributed inference
INFO 06-27 05:22:25 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-27 05:22:26 [core.py:455] Waiting for init message from front-end.
INFO 06-27 05:22:27 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=Non

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 06-27 05:22:32 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=4 pid=105294) INFO 06-27 05:22:32 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=4 pid=105294) INFO 06-27 05:22:32 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=5 pid=105295) INFO 06-27 05:22:32 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=0 pid=105290) INFO 06-27 05:22:32 [default_loader.py:272] Loading weights took 0.14 seconds
(VllmWorker rank=0 pid=105290) INFO 06-27 05:22:32 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=3 pid=105293) INFO 06-27 05:22:32 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=5 pid=105295) INFO 06-27 05:22:32 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=7 pid=105297) INFO 06-27 05:22:32 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=5 pid=105295) INFO 06-27

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

(VllmWorker rank=0 pid=105290) (VllmWorker rank=2 pid=105292) (VllmWorker rank=5 pid=105295) (VllmWorker rank=6 pid=105296) (VllmWorker rank=3 pid=105293) (VllmWorker rank=7 pid=105297) (VllmWorker rank=4 pid=105294) (VllmWorker rank=1 pid=105291) ERROR 06-27 05:23:30 [multiproc_executor.py:527] WorkerProc hit an exception.
ERROR 06-27 05:23:30 [multiproc_executor.py:527] WorkerProc hit an exception.
ERROR 06-27 05:23:30 [multiproc_executor.py:527] WorkerProc hit an exception.
(VllmWorker rank=0 pid=105290) ERROR 06-27 05:23:30 [multiproc_executor.py:527] WorkerProc hit an exception.
ERROR 06-27 05:23:30 [multiproc_executor.py:527] WorkerProc hit an exception.
ERROR 06-27 05:23:30 [multiproc_executor.py:527] WorkerProc hit an exception.
ERROR 06-27 05:23:30 [multiproc_executor.py:527] WorkerProc hit an exception.
(VllmWorker rank=2 pid=105292) ERROR 06-27 05:23:30 [multiproc_executor.py:527] WorkerProc hit an exception.
(VllmWorker rank=3 pid=105293) (VllmWorker rank=5 pid=105295) (Vll

...

INFO 06-27 05:25:52 [core.py:455] Waiting for init message from front-end.
INFO 06-27 05:25:52 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=0, served_model_name=meta-llama/Llama-3.2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


WARNING 06-27 05:25:53 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7fa9428ca770>
(VllmWorker rank=0 pid=116975) INFO 06-27 05:25:53 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_60482de7'), local_subscribe_addr='ipc:///tmp/83024716-94bf-4e9c-a3fb-ccb2d83eff51', remote_subscribe_addr=None, remote_addr_ipv6=False)
WARNING 06-27 05:25:53 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7fa9428ca470>
(VllmWorker rank=1 pid=116976) INFO 06-27 05:25:53 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_c44d43a3'), local_subscribe_addr='ipc:///tmp/b08dbb70-3926-439a-a693

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorker rank=6 pid=116984) INFO 06-27 05:25:57 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=4 pid=116979) INFO 06-27 05:25:57 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=3 pid=116978) INFO 06-27 05:25:57 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=3 pid=116978) INFO 06-27 05:25:57 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=1 pid=116976) INFO 06-27 05:25:57 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=0 pid=116975) INFO 06-27 05:25:57 [default_loader.py:272] Loading weights took 0.13 seconds
(VllmWorker rank=0 pid=116975) INFO 06-27 05:25:57 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=2 pid=116977) INFO 06-27 05:25:57 [weight_utils.py:345] No model.safetensors.index.json found in remote.
(VllmWorker rank=4 pid=116979) INFO 06-27 05:25:57 [default_loader.py:272] Loading weights took 0.12